# COMP2050 - Attention U-Net for Medical Image Segmentation
## Notebook 2: Train a Single Configuration

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# Configuration - change these to run different experiments
ARCHITECTURE = 'attention_unet'  # Options: unet, attention_unet, resunet, smp_resnet18
LOSS = 'bce_dice'                # Options: bce, dice, focal, bce_dice, tversky
SEED = 42
EPOCHS = 50
LR = 1e-4
BATCH_SIZE = 16
PATIENCE = 10
DATA_ROOT = './data'
OUTPUT_DIR = './results'

print(f'Config: {ARCHITECTURE} | {LOSS} | seed={SEED}')

In [ ]:
# Quick sanity check: verify model forward pass
from models import get_model
model = get_model(ARCHITECTURE)
x = torch.randn(2, 3, 256, 256)
y = model(x)
params = sum(p.numel() for p in model.parameters())
print(f'{ARCHITECTURE}: input={x.shape} -> output={y.shape}, params={params:,}')

In [ ]:
# Train
from train import train

config = {
    'architecture': ARCHITECTURE,
    'loss': LOSS,
    'seed': SEED,
    'epochs': EPOCHS,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'patience': PATIENCE,
    'data_root': DATA_ROOT,
    'output_dir': OUTPUT_DIR,
}

results = train(config)
print(f'\nFinal test Dice: {results["test_dice"]:.4f}')

In [ ]:
# Visualize training curves
import json, os
import matplotlib.pyplot as plt

run_name = f'{ARCHITECTURE}_{LOSS}_seed{SEED}'
with open(os.path.join(OUTPUT_DIR, run_name, 'history.json')) as f:
    history = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title(f'Training Loss ({run_name})')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_dice'], label='Val Dice', color='green')
ax2.plot(history['val_iou'], label='Val IoU', color='blue')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title(f'Validation Metrics ({run_name})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize segmentation results on test set
from data.dataset import KvasirSEGDataset
import numpy as np

model.eval()
model.cuda()

test_dataset = KvasirSEGDataset(DATA_ROOT, split='test', seed=SEED)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
indices = [0, len(test_dataset)//2, len(test_dataset)-1]

for row, idx in enumerate(indices):
    img, mask = test_dataset[idx]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).cuda())
        pred = torch.sigmoid(pred).squeeze().cpu().numpy()

    img_np = img.numpy().transpose(1, 2, 0)
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    mask_np = mask.squeeze().numpy()
    pred_bin = (pred > 0.5).astype(np.float32)

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Input')
    axes[row, 1].imshow(mask_np, cmap='gray')
    axes[row, 1].set_title('Ground Truth')
    axes[row, 2].imshow(pred, cmap='gray')
    axes[row, 2].set_title('Prediction (prob)')
    axes[row, 3].imshow(pred_bin, cmap='gray')
    axes[row, 3].set_title('Prediction (bin)')
    axes[row, 4].imshow(img_np)
    axes[row, 4].imshow(mask_np, cmap='Greens', alpha=0.4)
    axes[row, 4].set_title('GT Overlay')
    axes[row, 5].imshow(img_np)
    axes[row, 5].imshow(pred_bin, cmap='Reds', alpha=0.4)
    axes[row, 5].set_title('Pred Overlay')

    for c in range(6):
        axes[row, c].axis('off')

plt.suptitle(f'Segmentation Results: {run_name}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize attention maps (only for attention_unet)
if ARCHITECTURE == 'attention_unet':
    from utils.visualization import plot_attention_maps

    model.eval()
    idx = 0
    img, mask = test_dataset[idx]
    with torch.no_grad():
        _ = model(img.unsqueeze(0).cuda())
        att_maps = model.get_attention_maps()

    plot_attention_maps(img, att_maps)
else:
    print('Attention map visualization is only available for attention_unet.')